# 04 Parameters and Widgets

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Build <b>parameterized notebooks</b>: define widgets of every type, read and validate their values, turn them into a typed configuration, use them safely in Python and SQL (parameter markers and <code>IDENTIFIER</code>), and connect them to job parameters.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
A pipeline notebook should run unchanged for any date, environment or region. Parameters make that possible, and they're how Lakeflow Jobs, ADF and other orchestrators control notebooks. Hard-coded dates and table names are among the most common reasons notebooks can't be scheduled or promoted from dev to prod.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Widgets and job parameters</b><br>
| Widget | Creates | Read with |<br>|---|---|---|<br>| <code>dbutils.widgets.text(name, default, label)</code> | Free text box | <code>dbutils.widgets.get(name)</code> → string |<br>| <code>dbutils.widgets.dropdown(name, default, choices, label)</code> | Single choice | string |<br>| <code>dbutils.widgets.combobox(name, default, choices, label)</code> | Choice or free text | string |<br>| <code>dbutils.widgets.multiselect(name, default, choices, label)</code> | Several choices | comma-separated string |
<ul><li>Widget values are <b>always strings</b>: convert and validate them yourself</li><li>When a <b>job</b> runs the notebook, job and task parameters with the same names override the widget defaults</li><li>In SQL cells, widgets are available as <b>named parameter markers</b>: <code>:run_date</code></li></ul>
</div>

```
Job parameters {"run_date": "2024-11-05", "env": "prod"}
        │ fill widgets with the same names
        ▼
Notebook: widgets → read + validate → CONFIG dict → Python code and SQL (:params, IDENTIFIER)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A daily sales notebook started with <code>run_date = "2024-11-05"</code> hard-coded. Every morning someone edited it before running. With a <code>run_date</code> widget (defaulting to yesterday), an <code>env</code> dropdown that picks the catalog (<code>dev_sales</code>, <code>prod_sales</code>), and validation at the top, the same notebook now runs from a job for any date, in any environment, and backfills a month with a simple loop of job runs.
</div>

## 1. Create widgets of every type

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a text widget, a dropdown, a combobox and a multiselect at the top of the notebook.<br><br>
<b>Why it matters</b><br>Choose the widget type by what you want to allow: free values (text), a fixed list (dropdown), a list plus custom values (combobox), or several values (multiselect).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Four widgets appear above the first cell.
</div>

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("run_date", "2024-11-05", "Run date (yyyy-MM-dd)")
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"], "Environment")
dbutils.widgets.combobox("min_amount", "0", ["0", "50", "100"], "Minimum amount")
dbutils.widgets.multiselect("countries", "IN", ["IN", "UK", "AE", "US"], "Countries")

## 2. Read, convert and validate into a configuration

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads every widget, converts the values to proper types, validates them, and builds one configuration dictionary. The reading function falls back to defaults when widgets don't exist, so the same logic works in tests.<br><br>
<b>Why it matters</b><br>Validate parameters <b>at the top</b> of the notebook, with clear error messages. A wrong date format should fail in the first second, not halfway through a job after writing half the data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A configuration with a <code>date</code> object, the environment, a float minimum amount, a list of countries, and table names derived from the environment.
</div>

In [0]:
CATALOG = "workspace"
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.platform")

In [0]:
from datetime import date, datetime

DEFAULTS = {"run_date": "2024-11-05", "env": "dev", "min_amount": "0", "countries": "IN"}

def get_param(name):
    """Widget value on Databricks, default elsewhere (tests, local runs)."""
    try:
        return dbutils.widgets.get(name)        # noqa: F821
    except Exception:
        return DEFAULTS[name]

def build_config():
    raw = {name: get_param(name) for name in DEFAULTS}
    try:
        run_date = datetime.strptime(raw["run_date"], "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(f"run_date must be yyyy-MM-dd, got {raw['run_date']!r}")
    if raw["env"] not in ("dev", "test", "prod"):
        raise ValueError(f"unknown env {raw['env']!r}")
    min_amount = float(raw["min_amount"])
    if min_amount < 0:
        raise ValueError("min_amount must be >= 0")
    countries = [c for c in raw["countries"].split(",") if c]
    if not countries:
        raise ValueError("select at least one country")
    schema = "platform" if raw["env"] == "dev" else f"platform_{raw['env']}"
    return {
        "run_date": run_date,
        "env": raw["env"],
        "min_amount": min_amount,
        "countries": countries,
        "orders_table": f"{CATALOG}.{schema}.orders_params",
    }

CONFIG = build_config()
for k, v in CONFIG.items():
    print(f"{k:<13} {v!r}")

## 3. Use parameters in Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a small orders table and filters it with the configuration values.<br><br>
<b>Why it matters</b><br>Parameters flow into DataFrame code as normal Python values, already converted and validated.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The orders on 2024-11-05, in the selected countries, above the minimum amount.
</div>

In [0]:
from pyspark.sql import functions as F

spark.createDataFrame(
    [(1, "IN", "2024-11-05", 120.0), (2, "UK", "2024-11-05", 80.0), (3, "IN", "2024-11-04", 300.0),
     (4, "AE", "2024-11-05", 45.0), (5, "IN", "2024-11-05", 15.0)],
    "order_id INT, country STRING, order_date STRING, amount DOUBLE",
).withColumn("order_date", F.to_date("order_date")).write.mode("overwrite").saveAsTable(CONFIG["orders_table"])

result = (
    spark.table(CONFIG["orders_table"])
    .filter((F.col("order_date") == F.lit(CONFIG["run_date"]))
            & F.col("country").isin(CONFIG["countries"])
            & (F.col("amount") >= CONFIG["min_amount"]))
)
result.show()

## 4. Use parameters in SQL safely

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Passes the same parameters to SQL with <b>named parameter markers</b> and <code>IDENTIFIER()</code> for the table name.<br><br>
<b>Why it matters</b><br>Never format parameter values into SQL strings: quotes break the query, and it opens the door to SQL injection (lesson 01_26). Parameter markers pass values as data. <code>IDENTIFIER(:table)</code> safely turns a string into a table name. In a <code>%sql</code> cell in a Databricks notebook, widgets are available directly as <code>:run_date</code>, <code>:env</code> and so on.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same rows as section 3, from a parameterized SQL query.
</div>

In [0]:
spark.sql(
    """
    SELECT * FROM IDENTIFIER(:orders_table)
    WHERE order_date = :run_date
      AND array_contains(split(:countries, ','), country)
      AND amount >= :min_amount
    ORDER BY order_id
    """,
    args={
        "orders_table": CONFIG["orders_table"],
        "run_date": CONFIG["run_date"],
        "countries": ",".join(CONFIG["countries"]),
        "min_amount": CONFIG["min_amount"],
    },
).show()

```sql
-- The same in a %sql cell of a Databricks notebook: widgets are parameter markers
SELECT * FROM IDENTIFIER('workspace.platform.orders_params')
WHERE order_date = :run_date
  AND amount >= CAST(:min_amount AS DOUBLE)
```

## 5. Job parameters fill the widgets

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains how a job passes values to this notebook, and shows what the notebook sees.<br><br>
<b>Why it matters</b><br>In a Lakeflow Job, you define <b>job parameters</b> (shared by all tasks) or <b>task parameters</b>. When the notebook task runs, each parameter whose name matches a widget overrides that widget's default. Dynamic values such as <code>{{job.start_time.iso_date}}</code> let a job pass "today" automatically.
</div>

| Job / task parameter | Value in the job | Widget in the notebook |
|---|---|---|
| `run_date` | `{{job.start_time.iso_date}}` | `run_date` = the run's date |
| `env` | `prod` | `env` = `prod` |
| `countries` | `IN,UK` | `countries` = `IN,UK` |

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All current widget values as a dictionary. In a job run, they would show the job's parameter values.
</div>

In [0]:
try:
    print(dbutils.widgets.getAll())          # Databricks Runtime 15+ / serverless
except Exception:
    print({name: dbutils.widgets.get(name) for name in DEFAULTS})

## 6. Defaults that move with time: "yesterday"

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes a default run date of yesterday when the widget is empty, and runs a small backfill loop over several dates.<br><br>
<b>Why it matters</b><br>Daily jobs usually process "yesterday" by default, but must also accept an explicit date for reruns and backfills. Structuring the logic as a function of the date makes backfills a simple loop (or a job run per date).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Yesterday's date as the effective default, then one line per backfilled date with its row count.
</div>

In [0]:
from datetime import timedelta

def effective_run_date(value):
    return (date.today() - timedelta(days=1)) if value in (None, "", "yesterday") else datetime.strptime(value, "%Y-%m-%d").date()

print("default when empty:", effective_run_date(""))

def process(run_date):
    return spark.table(CONFIG["orders_table"]).filter(F.col("order_date") == F.lit(run_date)).count()

for d in [date(2024, 11, 4), date(2024, 11, 5)]:
    print(f"backfill {d}: {process(d)} orders")

## 7. Cleanup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Removes the widgets and the demo table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No widgets left.
</div>

In [0]:
dbutils.widgets.removeAll()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CONFIG['orders_table']}")

## Under the hood

```
Interactive run:  widget box values ───────────────────────────┐
Job run:          job/task parameters (same names) override ───┤
                                                               ▼
                         dbutils.widgets.get(name)  -> string   (Python)
                         :name                       -> value    (%sql cells)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> reading widgets costs nothing. It's the driver reading a value.

**Interview point:** widgets turn a notebook into a reusable function of its parameters. Combined with idempotent writes (`MERGE`, `replaceWhere`), the same notebook supports daily runs, reruns and backfills.

In [0]:
print("parameters in use:", {k: str(v) for k, v in CONFIG.items()})

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: comparing a widget value with a number fails</b><br>
Widget values are strings. Convert with <code>int()</code>, <code>float()</code> or <code>datetime.strptime()</code>.<br><br>
**⛔ Problem: <code>InputWidgetNotDefined</code>**<br>The widget wasn't created in this run, for example a job running the notebook without that parameter and without a <code>text()</code> call. Always create widgets (with defaults) before reading them.<br><br>
<b>⛔ Problem: SQL breaks when a value contains a quote</b><br>The value was formatted into the SQL string. Use parameter markers and <code>IDENTIFIER()</code>.<br><br>
<b>⛔ Problem: the job always processes the same date</b><br>The date is hard-coded or the widget default never changes. Pass <code>{{job.start_time.iso_date}}</code> (or compute "yesterday") as the parameter.<br><br>
<b>⛔ Problem: changing a widget doesn't re-run anything</b><br>Re-run the cells, or set the notebook's widget behaviour to "Run notebook" on change.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you parameterize a Databricks notebook?</b><br>
Create widgets with defaults at the top (<code>text</code>, <code>dropdown</code>, <code>combobox</code>, <code>multiselect</code>), read them with <code>dbutils.widgets.get</code>, convert and validate them into a config object, and use those values in the code. Jobs pass parameters with the same names, which override the defaults.<br><br>

<b>🎤 2. What type are widget values?</b><br>
Always strings, including multiselect, which is comma-separated. Convert them explicitly.<br><br>

<b>🎤 3. How do you use parameters in SQL safely?</b><br>
With named parameter markers (<code>:name</code>), passed with <code>args=</code> in <code>spark.sql</code> or taken from widgets in SQL cells, and <code>IDENTIFIER(:name)</code> for table or column names. Never string formatting.<br><br>

<b>🎤 4. How does a job pass values to a notebook?</b><br>
Through job or task parameters. Each one whose name matches a widget sets that widget's value for the run. Dynamic value references such as <code>{{job.start_time.iso_date}}</code> provide run-specific values.<br><br>

<b>🎤 5. How do you design a notebook for backfills?</b><br>
Make the processing date a parameter, make every write idempotent for that date (for example <code>replaceWhere</code> or <code>MERGE</code>), and run the notebook once per date, in a loop or as separate job runs.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook reads <code>n = dbutils.widgets.get("n")</code> and then fails on <code>n + 1</code>. Why?</b><br>
A. Widgets can't be read in Python<br>
B. Widget values are strings and must be converted, for example <code>int(n)</code><br>
C. The widget must be a dropdown<br>
D. <code>dbutils</code> is not available in notebooks<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A notebook task in a job must process the date on which the job run started. What is the best way to provide it?</b><br>
A. Hard-code the date and edit it daily<br>
B. A task parameter <code>run_date</code> set to <code>{{job.start_time.iso_date}}</code>, read by a widget with the same name<br>
C. Read the date from the driver logs<br>
D. Use <code>spark.conf.get("run_date")</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a widget <code>mode</code> (dropdown: <code>full</code>, <code>incremental</code>) and extend <code>build_config()</code> to validate it</li><li>When <code>mode = full</code>, ignore <code>run_date</code> and process all dates. When <code>incremental</code>, process only <code>run_date</code></li><li>Write the SQL version of the filter using parameter markers, including <code>IDENTIFIER</code> for the table</li><li>Make <code>run_date</code> default to yesterday when the widget is left empty</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. New parameter with validation (works with or without widgets)
DEFAULTS["mode"] = "incremental"
def build_config_v2():
    cfg = build_config()
    mode = get_param("mode")
    if mode not in ("full", "incremental"):
        raise ValueError(f"mode must be full or incremental, got {mode!r}")
    cfg["mode"] = mode
    # 4. Empty run_date means yesterday
    cfg["run_date"] = effective_run_date(get_param("run_date")) if get_param("run_date") else effective_run_date("")
    return cfg

cfg = build_config_v2()
print(cfg)

# Recreate the demo table for this exercise
spark.createDataFrame([(1, "IN", "2024-11-05", 120.0), (2, "IN", "2024-11-04", 80.0)],
                      "order_id INT, country STRING, order_date STRING, amount DOUBLE") \
    .withColumn("order_date", F.to_date("order_date")).write.mode("overwrite").saveAsTable(cfg["orders_table"])

# 2. Full vs incremental
df = spark.table(cfg["orders_table"])
if cfg["mode"] == "incremental":
    df = df.filter(F.col("order_date") == F.lit(cfg["run_date"]))
print("rows to process:", df.count())

# 3. SQL with markers
spark.sql("SELECT count(*) AS n FROM IDENTIFIER(:t) WHERE :mode = 'full' OR order_date = :d",
          args={"t": cfg["orders_table"], "mode": cfg["mode"], "d": cfg["run_date"]}).show()
spark.sql(f"DROP TABLE {cfg['orders_table']}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Widgets: <code>text</code>, <code>dropdown</code>, <code>combobox</code>, <code>multiselect</code>. Values are always strings</li><li>Read, convert and validate parameters at the top into one config object</li><li>Job and task parameters with matching names override widget defaults</li><li>In SQL: <code>:name</code> parameter markers and <code>IDENTIFIER(:table)</code>. Never string formatting</li><li>Use dynamic values such as <code>{{job.start_time.iso_date}}</code>, or a "yesterday" default</li><li>Parameterized + idempotent = safe reruns and easy backfills</li></ul>
</div>

| Task | Code |
|---|---|
| Text widget | `dbutils.widgets.text("run_date", "2024-11-05")` |
| Dropdown | `dbutils.widgets.dropdown("env", "dev", ["dev", "prod"])` |
| Read | `dbutils.widgets.get("env")` |
| All values | `dbutils.widgets.getAll()` |
| SQL marker | `WHERE d = :run_date`, `FROM IDENTIFIER(:table)` |
| Remove | `dbutils.widgets.removeAll()` |

## Git commit

```
git add 04_databricks/04_parameters_and_widgets.ipynb
git commit -m "add 04_04 parameters and widgets notebook"
```